# Like-Day Model Forecast Investigation
## PJM Western Hub DA LMP -- 2026-02-26

**Objective:** Investigate the like-day model forecast for 2026-02-26.  
The model produced MAE $5.48, RMSE $7.40, with systematic underprediction of evening peaks.  
This notebook analyzes the forecast, analog selection, and sensitivity to key parameters.

## 1. Setup & Run Forecast

In [ ]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 100,
})

# Initialize settings (loads .env, configures logging)
import src.like_day_forecast.settings

from src.like_day_forecast.pipelines.forecast import run
from src.like_day_forecast import configs

FORECAST_DATE = "2026-02-26"
HOURS = list(range(1, 25))
HE_LABELS = [f"HE{h}" for h in HOURS]
ONPEAK_HOURS = list(range(8, 24))   # HE8-HE23
OFFPEAK_HOURS = list(range(1, 8)) + [24]

In [ ]:
# Run the forecast pipeline
result = run(forecast_date=FORECAST_DATE)

# Unpack key objects
output_table = result["output_table"]
quantiles_table = result["quantiles_table"]
analogs_df = result["analogs"]
metrics_dict = result["metrics"]
df_forecast = result["df_forecast"]

print(f"\nForecast date:  {result['forecast_date']}")
print(f"Reference date: {result['reference_date']}")
print(f"Analogs used:   {result['n_analogs_used']}")
print(f"Has actuals:    {result['has_actuals']}")

## 2. Forecast vs Actuals -- Hourly Profile Plot

Line chart comparing actual DA LMP to the point forecast by hour ending, with shaded prediction interval bands (80% PI and 90% PI). Hours where the actual falls outside the 90% PI are highlighted.

In [ ]:
# Extract hourly series from the output_table
actual_row = output_table[output_table["Type"] == "Actual"]
forecast_row = output_table[output_table["Type"] == "Forecast"]

actuals = np.array([actual_row[f"HE{h}"].values[0] for h in HOURS])
forecasts = np.array([forecast_row[f"HE{h}"].values[0] for h in HOURS])

# Quantile bands from df_forecast
p05 = df_forecast["q_0.05"].values
p10 = df_forecast["q_0.10"].values
p90 = df_forecast["q_0.90"].values
p95 = df_forecast["q_0.95"].values

fig, ax = plt.subplots(figsize=(14, 6))

# 90% PI band (P05-P95)
ax.fill_between(HOURS, p05, p95, alpha=0.15, color="steelblue", label="90% PI (P05-P95)")
# 80% PI band (P10-P90)
ax.fill_between(HOURS, p10, p90, alpha=0.25, color="steelblue", label="80% PI (P10-P90)")

# Forecast and actual lines
ax.plot(HOURS, forecasts, "o-", color="steelblue", linewidth=2, markersize=5, label="Forecast")
ax.plot(HOURS, actuals, "s-", color="firebrick", linewidth=2, markersize=5, label="Actual")

# Highlight hours where actuals fall outside the 90% PI
outside_90 = (actuals < p05) | (actuals > p95)
if outside_90.any():
    ax.scatter(
        np.array(HOURS)[outside_90], actuals[outside_90],
        s=120, facecolors="none", edgecolors="red", linewidths=2.5,
        zorder=5, label="Outside 90% PI",
    )

ax.set_xlabel("Hour Ending")
ax.set_ylabel("DA LMP ($/MWh)")
ax.set_title(f"Like-Day Forecast vs Actuals -- Western Hub DA LMP -- {FORECAST_DATE}")
ax.set_xticks(HOURS)
ax.set_xticklabels(HE_LABELS, rotation=45, fontsize=9)
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

mae_val = metrics_dict.get("mae", np.nan)
rmse_val = metrics_dict.get("rmse", np.nan)
ax.text(
    0.98, 0.97, f"MAE: ${mae_val:.2f}/MWh\nRMSE: ${rmse_val:.2f}/MWh",
    transform=ax.transAxes, ha="right", va="top",
    fontsize=11, bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8),
)

plt.tight_layout()
plt.show()

## 3. Error Analysis

Bar chart of hourly forecast errors (Forecast - Actual), with on-peak vs off-peak and peak-hour (HE16-HE22) metrics broken out separately.

In [ ]:
# Hourly error: Forecast - Actual (positive = overpredicted, negative = underpredicted)
errors = forecasts - actuals

# Color: green for underpredict (negative), red for overpredict (positive)
colors = ["#d32f2f" if e > 0 else "#388e3c" for e in errors]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(HOURS, errors, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Hour Ending")
ax.set_ylabel("Forecast Error ($/MWh)")
ax.set_title(f"Hourly Forecast Error (Forecast - Actual) -- {FORECAST_DATE}")
ax.set_xticks(HOURS)
ax.set_xticklabels(HE_LABELS, rotation=45, fontsize=9)

# Add value labels on bars
for h, e in zip(HOURS, errors):
    ax.text(h, e + (0.3 if e >= 0 else -0.3), f"{e:.1f}",
            ha="center", va="bottom" if e >= 0 else "top", fontsize=7)

# Legend patch
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#388e3c", label="Underpredict (Forecast < Actual)"),
    Patch(facecolor="#d32f2f", label="Overpredict (Forecast > Actual)"),
]
ax.legend(handles=legend_elements, loc="upper left")

plt.tight_layout()
plt.show()

# --- Compute segmented metrics ---
onpeak_mask = np.array([h in ONPEAK_HOURS for h in HOURS])
offpeak_mask = np.array([h in OFFPEAK_HOURS for h in HOURS])
peak_mask = np.array([h in range(16, 23) for h in HOURS])  # HE16-HE22

def segment_metrics(mask, label):
    seg_err = errors[mask]
    seg_act = actuals[mask]
    seg_fct = forecasts[mask]
    seg_mae = np.mean(np.abs(seg_err))
    seg_rmse = np.sqrt(np.mean(seg_err ** 2))
    seg_bias = np.mean(seg_err)
    print(f"  {label:<18s}  MAE: ${seg_mae:.2f}  |  RMSE: ${seg_rmse:.2f}  |  Bias: ${seg_bias:+.2f}")

print(f"\nSegmented Error Metrics for {FORECAST_DATE}:")
print("-" * 65)
segment_metrics(np.ones(24, dtype=bool), "All Hours")
segment_metrics(onpeak_mask, "On-Peak (HE8-23)")
segment_metrics(offpeak_mask, "Off-Peak (HE1-7,24)")
segment_metrics(peak_mask, "Peak (HE16-22)")

## 4. Analog Day Analysis

Top 10 analog days with their distance, weight, and similarity. Then overlay the top 5 analogs' next-day hourly DA LMP profiles against the actual 2026-02-26 profile to see which analogs drive the forecast.

In [ ]:
# Top 10 analogs table
top10 = analogs_df.head(10).copy()
top10["date"] = top10["date"].astype(str)
top10["distance"] = top10["distance"].map("{:.4f}".format)
top10["weight"] = top10["weight"].map("{:.4f}".format)
top10["similarity"] = top10["similarity"].map("{:.2%}".format)

display(top10[["rank", "date", "distance", "weight", "similarity"]])

In [ ]:
# Pull raw LMP data and plot top-5 analog next-day profiles vs actual
from src.like_day_forecast.data import lmps_hourly
from datetime import timedelta

df_lmp_all = lmps_hourly.pull(schema=configs.SCHEMA, hub=configs.HUB, market="da")

target_date = pd.to_datetime(FORECAST_DATE).date()

# Actual profile for forecast date
df_actual = df_lmp_all[df_lmp_all["date"] == target_date].sort_values("hour_ending")
actual_profile = df_actual["lmp_total"].values

# Top 5 analogs — get their NEXT DAY profiles
top5_analogs = analogs_df.head(5)
cmap = plt.cm.tab10

fig, ax = plt.subplots(figsize=(14, 6))

for i, (_, row) in enumerate(top5_analogs.iterrows()):
    analog_date = row["date"]
    next_date = analog_date + timedelta(days=1)
    df_next = df_lmp_all[df_lmp_all["date"] == next_date].sort_values("hour_ending")
    if len(df_next) >= 24:
        ax.plot(
            HOURS, df_next["lmp_total"].values,
            "--", color=cmap(i), alpha=0.7, linewidth=1.5,
            label=f"Analog {row['rank']:.0f}: {analog_date} (next: {next_date}, w={row['weight']:.3f})",
        )

ax.plot(HOURS, actual_profile, "s-", color="firebrick", linewidth=2.5, markersize=5,
        label=f"Actual {FORECAST_DATE}", zorder=10)
ax.plot(HOURS, forecasts, "o-", color="steelblue", linewidth=2, markersize=4,
        label="Forecast (weighted avg)", zorder=9)

ax.set_xlabel("Hour Ending")
ax.set_ylabel("DA LMP ($/MWh)")
ax.set_title(f"Top 5 Analog Next-Day LMP Profiles vs Actual -- {FORECAST_DATE}")
ax.set_xticks(HOURS)
ax.set_xticklabels(HE_LABELS, rotation=45, fontsize=9)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Analog Date Distribution

Histogram of analog dates by year and scatter plot of analog date vs distance (colored by weight) to reveal temporal clustering.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Histogram by year ---
analog_dates_dt = pd.to_datetime(analogs_df["date"])
years = analog_dates_dt.dt.year

ax = axes[0]
year_counts = years.value_counts().sort_index()
ax.bar(year_counts.index.astype(str), year_counts.values, color="steelblue", edgecolor="white")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Analogs")
ax.set_title("Analog Days by Year")
ax.tick_params(axis="x", rotation=45)

for x_val, y_val in zip(year_counts.index.astype(str), year_counts.values):
    ax.text(x_val, y_val + 0.3, str(y_val), ha="center", va="bottom", fontsize=10)

# --- Right: Scatter — date vs distance, colored by weight ---
ax = axes[1]
scatter = ax.scatter(
    analog_dates_dt, analogs_df["distance"],
    c=analogs_df["weight"], cmap="YlOrRd", s=60, edgecolors="gray", linewidths=0.5,
)
ax.set_xlabel("Analog Date")
ax.set_ylabel("Distance")
ax.set_title("Analog Date vs Distance (colored by weight)")
fig.colorbar(scatter, ax=ax, label="Weight", shrink=0.8)
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print(f"\nAnalog date range: {analog_dates_dt.min().date()} to {analog_dates_dt.max().date()}")
print(f"Median analog year: {int(years.median())}")
print(f"Most common year: {year_counts.idxmax()} ({year_counts.max()} analogs)")

## 6. Feature Sensitivity

Test the forecast with different `n_analogs` values and different weighting methods to see how the choice of hyperparameters affects forecast accuracy.

In [ ]:
# --- 6a: Sweep n_analogs ---
n_analogs_values = [10, 15, 20, 25, 30, 40, 50]
n_results = []

for n in n_analogs_values:
    r = run(forecast_date=FORECAST_DATE, n_analogs=n, weight_method="inverse_distance")
    if r.get("metrics"):
        n_results.append({
            "n_analogs": n,
            "MAE": r["metrics"]["mae"],
            "RMSE": r["metrics"]["rmse"],
            "MAPE": r["metrics"].get("mape", np.nan),
        })
    print(f"  n_analogs={n:3d}  ->  MAE: ${r['metrics']['mae']:.2f}  RMSE: ${r['metrics']['rmse']:.2f}")

df_n = pd.DataFrame(n_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df_n["n_analogs"], df_n["MAE"], "o-", color="steelblue", linewidth=2, label="MAE")
ax.plot(df_n["n_analogs"], df_n["RMSE"], "s-", color="firebrick", linewidth=2, label="RMSE")
ax.set_xlabel("Number of Analogs")
ax.set_ylabel("Error ($/MWh)")
ax.set_title("Forecast Error vs Number of Analogs")
ax.set_xticks(n_analogs_values)
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate minimum
best_n_mae = df_n.loc[df_n["MAE"].idxmin()]
ax.annotate(
    f"Best MAE: ${best_n_mae['MAE']:.2f}\n(n={int(best_n_mae['n_analogs'])})",
    xy=(best_n_mae["n_analogs"], best_n_mae["MAE"]),
    xytext=(best_n_mae["n_analogs"] + 5, best_n_mae["MAE"] + 0.5),
    arrowprops=dict(arrowstyle="->", color="steelblue"),
    fontsize=10, color="steelblue",
)

plt.tight_layout()
plt.show()

In [ ]:
# --- 6b: Sweep weight methods ---
weight_methods = ["inverse_distance", "softmax", "rank", "uniform"]
w_results = []

for method in weight_methods:
    r = run(forecast_date=FORECAST_DATE, n_analogs=30, weight_method=method)
    if r.get("metrics"):
        w_results.append({
            "method": method,
            "MAE": r["metrics"]["mae"],
            "RMSE": r["metrics"]["rmse"],
            "MAPE": r["metrics"].get("mape", np.nan),
        })
    print(f"  {method:<20s}  ->  MAE: ${r['metrics']['mae']:.2f}  RMSE: ${r['metrics']['rmse']:.2f}")

df_w = pd.DataFrame(w_results)

fig, ax = plt.subplots(figsize=(8, 5))
x_pos = range(len(df_w))
bars_mae = ax.bar([x - 0.15 for x in x_pos], df_w["MAE"], 0.3,
                   label="MAE", color="steelblue", edgecolor="white")
bars_rmse = ax.bar([x + 0.15 for x in x_pos], df_w["RMSE"], 0.3,
                    label="RMSE", color="firebrick", edgecolor="white")

ax.set_xlabel("Weight Method")
ax.set_ylabel("Error ($/MWh)")
ax.set_title("Forecast Error by Weighting Method (n_analogs=30)")
ax.set_xticks(list(x_pos))
ax.set_xticklabels(df_w["method"], rotation=20)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Annotate bars
for bar in bars_mae:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"${bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
for bar in bars_rmse:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f"${bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Season Window Sensitivity

Test different `season_window_days` values by calling `find_analogs` directly with a pre-built feature matrix. This controls how wide the seasonal filter is, which directly affects the analog candidate pool size and quality.

In [ ]:
from src.like_day_forecast.similarity.engine import find_analogs
from src.like_day_forecast.features.builder import build_daily_features

# Build features once
df_features = build_daily_features()
reference_date = pd.to_datetime(FORECAST_DATE).date() - timedelta(days=1)

season_windows = [30, 45, 60, 90]
sw_results = []

for sw in season_windows:
    analogs = find_analogs(
        target_date=reference_date,
        df_features=df_features,
        n_analogs=30,
        season_window_days=sw,
        weight_method="inverse_distance",
    )
    sw_results.append({
        "season_window": sw,
        "n_analogs_found": len(analogs),
        "min_distance": analogs["distance"].min(),
        "median_distance": analogs["distance"].median(),
        "max_distance": analogs["distance"].max(),
        "mean_distance": analogs["distance"].mean(),
        "top5_mean_dist": analogs.head(5)["distance"].mean(),
    })
    print(f"  window={sw:3d}d  pool->30 analogs  "
          f"dist range: {analogs['distance'].min():.4f} - {analogs['distance'].max():.4f}  "
          f"top-5 mean: {analogs.head(5)['distance'].mean():.4f}")

df_sw = pd.DataFrame(sw_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: top-5 mean distance vs window
ax = axes[0]
ax.plot(df_sw["season_window"], df_sw["top5_mean_dist"], "o-", color="steelblue", linewidth=2, label="Top-5 Mean")
ax.plot(df_sw["season_window"], df_sw["mean_distance"], "s--", color="firebrick", linewidth=2, label="All-30 Mean")
ax.set_xlabel("Season Window (days)")
ax.set_ylabel("Distance")
ax.set_title("Analog Distance vs Season Window Size")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: distance range (min/max) vs window
ax = axes[1]
ax.fill_between(df_sw["season_window"], df_sw["min_distance"], df_sw["max_distance"],
                alpha=0.3, color="steelblue", label="Distance Range")
ax.plot(df_sw["season_window"], df_sw["median_distance"], "o-", color="steelblue",
        linewidth=2, label="Median Distance")
ax.set_xlabel("Season Window (days)")
ax.set_ylabel("Distance")
ax.set_title("Analog Distance Range by Season Window")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary Statistics Table

Full metrics dictionary, output table (Actual/Forecast/Error), and quantile bands.

In [ ]:
# Display full metrics
print("Full Evaluation Metrics")
print("=" * 50)
if metrics_dict:
    for k, v in metrics_dict.items():
        if isinstance(v, float):
            print(f"  {k:<25s}: {v:.4f}")
        else:
            print(f"  {k:<25s}: {v}")
else:
    print("  No metrics available (actuals may be missing).")

In [ ]:
# Output table: Actual / Forecast / Error
print("Output Table ($/MWh)")
print("=" * 50)
display(output_table.style.format(
    {col: "{:.2f}" for col in output_table.columns if col not in ("Date", "Type")},
    na_rep="--",
))

In [ ]:
# Quantiles table
print("Quantile Bands ($/MWh)")
print("=" * 50)
display(quantiles_table.style.format(
    {col: "{:.2f}" for col in quantiles_table.columns if col not in ("Date", "Type")},
    na_rep="--",
))